# TP Integrador - Preparación de datos, fine-tuning y evaluación

**Problema:** clasificación de imágenes de Pokémon de primera generación según su tipo principal traducido (`type1`).

Este notebook adapta el flujo de trabajo a lo pedido para el repositorio del TP:

- descarga reproducible del dataset, sin subir imágenes al repo;
- análisis del dataset y distribución por clases;
- creación de `train.csv`, `val.csv` y `test.csv` versionables;
- `Dataset` y `DataLoader` de PyTorch;
- preprocesamiento, normalización y data augmentation solo en entrenamiento;
- fine-tuning de modelos preentrenados con al menos 3 configuraciones;
- curvas de entrenamiento, métricas finales en test, matriz de confusión y análisis de errores;
- guardado del modelo final como `dev/modelo.pth`.

> Nota: este notebook deja listo el modelo y la documentación base. La app integrada se encuentra en `prod/` y carga `dev/modelo.pth`.


## 1. Instalación de dependencias

En Colab o una máquina nueva, ejecutar esta celda para instalar las librerías necesarias. En local, también se puede usar el `prod/requirements.txt` del repo o crear un entorno virtual propio.


In [ ]:
!pip install -q kagglehub pandas matplotlib numpy pillow scikit-learn torch torchvision tqdm


## 2. Imports, configuración y reproducibilidad

Se fija una semilla para que la partición train/val/test sea reproducible y todos los integrantes trabajen con los mismos splits.


In [ ]:
from pathlib import Path
import os
import shutil
import random
import json
from glob import glob

import kagglehub
import numpy as np
pandas_import_error = None
try:
    import pandas as pd
except Exception as e:
    pandas_import_error = e
    raise

import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

SEED = 251
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

# Detectar raíz del repositorio tanto si el notebook se corre desde /dev como desde la raíz.
ROOT = Path.cwd()
if ROOT.name == "dev":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DEV_DIR = ROOT / "dev"

for path in [DATA_DIR, RAW_DIR, DEV_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("RAW_DIR:", RAW_DIR)


## 3. Descarga automática del dataset

Se usan dos fuentes públicas de Kaggle:

1. `mlomuscio/pokemon`: información tabular de los Pokémon, incluyendo generación y tipo principal.
2. `lantian773030/pokemonclassification`: imágenes organizadas por Pokémon.

Las imágenes quedan dentro de `data/raw/`, carpeta que debe estar ignorada por Git. Los CSV de splits sí se versionan.


In [ ]:
def copy_tree_contents(src: Path, dst: Path):
    """Copia el contenido de un directorio descargado por kagglehub a data/raw sin duplicar si ya existe."""
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        target = dst / item.name
        if target.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)

def download_kaggle_dataset(handle: str, destination_name: str) -> Path:
    """Descarga un dataset de Kaggle con kagglehub y lo copia a data/raw/<destination_name>."""
    destination = RAW_DIR / destination_name
    if destination.exists() and any(destination.iterdir()):
        print(f"Dataset ya disponible en {destination}")
        return destination

    print(f"Descargando {handle}...")
    downloaded_path = Path(kagglehub.dataset_download(handle))
    copy_tree_contents(downloaded_path, destination)
    print(f"Copiado en {destination}")
    return destination

TABULAR_DIR = download_kaggle_dataset("mlomuscio/pokemon", "pokemon_tabular")
IMAGES_DIR = download_kaggle_dataset("lantian773030/pokemonclassification", "pokemon_images")

print("Archivos tabulares encontrados:", list(TABULAR_DIR.rglob("*.csv"))[:5])
print("Carpetas de imágenes encontradas:", [p for p in IMAGES_DIR.rglob("*") if p.is_dir()][:5])


## 4. Carga y limpieza de datos tabulares

Se trabaja con Pokémon de primera generación para mantener un problema acotado. La etiqueta de clasificación es el tipo principal traducido (`type1`).


In [ ]:
def normalize_name(name: str) -> str:
    return (
        str(name).lower()
        .replace(".", "")
        .replace("'", "")
        .replace("♀", "")
        .replace("♂", "")
        .replace(" ", "")
        .replace("-", "")
    )


TYPE_TRANSLATION = {
    "Bug": "Bicho",
    "Dragon": "Dragon",
    "Electric": "Electrico",
    "Fairy": "Hada",
    "Fighting": "Lucha",
    "Fire": "Fuego",
    "Ghost": "Fantasma",
    "Grass": "Planta",
    "Ground": "Tierra",
    "Ice": "Hielo",
    "Normal": "Normal",
    "Poison": "Veneno",
    "Psychic": "Psiquico",
    "Rock": "Roca",
    "Water": "Agua",
}

csv_candidates = list(TABULAR_DIR.rglob("*.csv"))
if not csv_candidates:
    raise FileNotFoundError("No se encontró ningún CSV en el dataset tabular.")

CSV_PATH = csv_candidates[0]
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.lower()

required_columns = {"name", "type1", "generation"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas requeridas en el CSV: {missing}")

df_gen1 = df[df["generation"] == 1].copy()

# Algunas versiones del dataset traen columna num, otras pueden usar #.
if "num" not in df_gen1.columns:
    possible_num_cols = [c for c in df_gen1.columns if c in ["#", "number", "pokedex_number"]]
    if possible_num_cols:
        df_gen1["num"] = df_gen1[possible_num_cols[0]]
    else:
        df_gen1["num"] = np.arange(1, len(df_gen1) + 1)

df_gen1 = df_gen1[["name", "type1", "num"]].copy()
df_gen1["normalized_name"] = df_gen1["name"].apply(normalize_name)

# Limpieza de nombres especiales y variantes que pueden romper el mapeo con carpetas.
df_gen1["name"] = df_gen1["name"].replace({"Mr. Mime": "MrMime", "Farfetch'd": "Farfetchd"})
df_gen1 = df_gen1[~df_gen1["name"].astype(str).str.contains("Mega", na=False)]
df_gen1 = df_gen1.drop_duplicates(subset=["normalized_name"])

print("Cantidad de Pokémon de Gen 1 luego de limpieza:", len(df_gen1))
df_gen1.head()


## 5. Mapeo de imágenes con etiquetas

El dataset de imágenes viene organizado por carpetas con el nombre del Pokémon. Se recorren esas carpetas y se asigna el `type1` correspondiente desde el CSV.


In [ ]:
# Buscar automáticamente la carpeta que contiene subcarpetas por Pokémon.
def find_images_root(base_dir: Path) -> Path:
    image_extensions = {".jpg", ".jpeg", ".png"}
    candidate_dirs = []
    for directory in [base_dir] + [p for p in base_dir.rglob("*") if p.is_dir()]:
        child_dirs = [p for p in directory.iterdir() if p.is_dir()]
        image_count = sum(1 for p in directory.rglob("*") if p.suffix.lower() in image_extensions)
        if len(child_dirs) > 20 and image_count > 100:
            candidate_dirs.append((directory, len(child_dirs), image_count))
    if not candidate_dirs:
        raise FileNotFoundError("No se pudo detectar la carpeta raíz de imágenes.")
    candidate_dirs.sort(key=lambda x: (x[1], x[2]), reverse=True)
    return candidate_dirs[0][0]

IMAGES_ROOT = find_images_root(IMAGES_DIR)
print("Carpeta raíz de imágenes:", IMAGES_ROOT)

valid_extensions = (".jpg", ".jpeg", ".png")
image_data = []
not_matched = []

for folder_path in sorted([p for p in IMAGES_ROOT.iterdir() if p.is_dir()]):
    folder_name = folder_path.name
    normalized_folder = normalize_name(folder_name)
    pokemon_match = df_gen1[df_gen1["normalized_name"] == normalized_folder]

    if pokemon_match.empty:
        not_matched.append(folder_name)
        continue

    pokemon_name = pokemon_match.iloc[0]["name"]
    pokemon_type_english = pokemon_match.iloc[0]["type1"]
    pokemon_type = TYPE_TRANSLATION.get(pokemon_type_english, pokemon_type_english)

    for image_path in folder_path.rglob("*"):
        if image_path.suffix.lower() not in valid_extensions:
            continue
        relative_path = image_path.relative_to(ROOT).as_posix()
        image_data.append({
            "pokemon": pokemon_name,
            "type1": pokemon_type,
            "type1_original": pokemon_type_english,
            "image_path": relative_path,
        })

final_df = pd.DataFrame(image_data)
if final_df.empty:
    raise ValueError("No se pudieron mapear imágenes con etiquetas.")

print(f"Total de imágenes mapeadas: {len(final_df)}")
print(f"Carpetas no mapeadas: {len(not_matched)}")
final_df.head()


## 6. Análisis del dataset

Se informa cantidad total de imágenes, clases, distribución por clase y ejemplos reales. Esto permite detectar desbalance y revisar la calidad visual del dataset.


In [ ]:
print("Cantidad total de imágenes:", len(final_df))
print("Cantidad de clases:", final_df["type1"].nunique())
print("Clases:", sorted(final_df["type1"].unique()))

class_distribution = final_df["type1"].value_counts().sort_values(ascending=False)
distribution_df = class_distribution.rename_axis("type1").reset_index(name="cantidad")
distribution_df


In [ ]:
plt.figure(figsize=(12, 5))
class_distribution.plot(kind="bar")
plt.title("Distribución de imágenes por tipo principal")
plt.xlabel("Tipo principal")
plt.ylabel("Cantidad de imágenes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Revisar resolución y relación de aspecto en una muestra para describir el dataset.
sample_paths = final_df["image_path"].sample(min(200, len(final_df)), random_state=SEED).tolist()
image_stats = []
for rel_path in sample_paths:
    path = ROOT / rel_path
    try:
        with Image.open(path) as img:
            w, h = img.size
            image_stats.append({"width": w, "height": h, "aspect_ratio": round(w / h, 3)})
    except Exception:
        pass

stats_df = pd.DataFrame(image_stats)
stats_df.describe()


In [ ]:
def show_dataset_examples(df_source, n=10):
    sample = df_source.sample(min(n, len(df_source)), random_state=SEED).reset_index(drop=True)
    cols = 5
    rows = int(np.ceil(len(sample) / cols))
    plt.figure(figsize=(15, 3 * rows))
    for i, row in sample.iterrows():
        image = Image.open(ROOT / row["image_path"]).convert("RGB")
        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(f'{row["pokemon"]}
Tipo: {row["type1"]}')
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_dataset_examples(final_df, n=10)


## 7. Particionado train / validación / test

Se divide el dataset en 70% entrenamiento, 15% validación y 15% test. La partición es estratificada por clase para conservar la distribución de tipos en cada conjunto.


In [ ]:
train_df, temp_df = train_test_split(
    final_df,
    test_size=0.30,
    stratify=final_df["type1"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["type1"],
    random_state=SEED,
)

# Guardar CSV livianos y versionables en data/.
train_df.to_csv(DATA_DIR / "train.csv", index=False)
val_df.to_csv(DATA_DIR / "val.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

print("\nDistribución train")
print(train_df["type1"].value_counts())
print("\nDistribución val")
print(val_df["type1"].value_counts())
print("\nDistribución test")
print(test_df["type1"].value_counts())


## 8. Transformaciones y normalización

El modelo preentrenado espera imágenes de 224x224 y normalización ImageNet. Las augmentations se aplican **solo a train**, no a validación ni a test.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.25, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

classes = sorted(final_df["type1"].unique())
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

print("Mapeo de clases:")
print(class_to_idx)


## 9. Visualización de data augmentation

Se muestran varias versiones aumentadas de imágenes del conjunto de entrenamiento para comprobar que las transformaciones tienen sentido para el problema.


In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    tensor = tensor.cpu() * std + mean
    return tensor.clamp(0, 1)

def show_augmentations(df_source, images_to_show=4, versions=4):
    sample = df_source.sample(min(images_to_show, len(df_source)), random_state=SEED).reset_index(drop=True)
    plt.figure(figsize=(versions * 3, len(sample) * 3))
    for row_idx, row in sample.iterrows():
        image = Image.open(ROOT / row["image_path"]).convert("RGB")
        for version in range(versions):
            aug = train_transform(image)
            aug = denormalize(aug).permute(1, 2, 0).numpy()
            ax_idx = row_idx * versions + version + 1
            plt.subplot(len(sample), versions, ax_idx)
            plt.imshow(aug)
            plt.title(f'{row["pokemon"]} - {row["type1"]}')
            plt.axis("off")
    plt.tight_layout()
    plt.show()

show_augmentations(train_df, images_to_show=4, versions=4)


## 10. Dataset personalizado y DataLoader

Se implementa una clase propia que hereda de `torch.utils.data.Dataset`. Esto permite leer desde los CSV de splits y mantener el particionado reproducible.


In [ ]:
class PokemonTypeDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = ROOT / row["image_path"]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[row["type1"]]
        return image, label

BATCH_SIZE = 32

train_dataset = PokemonTypeDataset(DATA_DIR / "train.csv", transform=train_transform)
val_dataset = PokemonTypeDataset(DATA_DIR / "val.csv", transform=eval_transform)
test_dataset = PokemonTypeDataset(DATA_DIR / "test.csv", transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print("Shape del batch:", images.shape)
print("Shape de labels:", labels.shape)
print("Rango luego de normalización:", float(images.min()), float(images.max()))


## 11. Verificación final de un batch

Se desnormalizan imágenes de un batch para visualizar que los pares imagen-etiqueta sean correctos.


In [ ]:
def show_batch(images, labels, n=8):
    n = min(n, len(images))
    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(12, 3 * rows))
    for i in range(n):
        image = denormalize(images[i]).permute(1, 2, 0).numpy()
        label = idx_to_class[int(labels[i])]
        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(label)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_batch(images, labels, n=8)


## 12. Modelo preentrenado y estrategia de fine-tuning

Se prueban modelos preentrenados de `torchvision`. En todos los casos se reemplaza la capa final para que la salida coincida con la cantidad de tipos de Pokémon.

Estrategias:

- `fc`: se congelan las capas convolucionales y se entrena solo la capa final.
- `layer4`: se descongela el último bloque de ResNet y la capa final.
- `lastfeature`: para EfficientNet, se descongela el último bloque de features y el clasificador.

Como el dataset no es muy grande, no se entrena desde cero: se aprovechan representaciones generales aprendidas en ImageNet.


In [ ]:
num_classes = len(classes)

def create_model(architecture="resnet18", fine_tuning="fc"):
    if architecture == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
    elif architecture == "resnet34":
        weights = models.ResNet34_Weights.DEFAULT
        model = models.resnet34(weights=weights)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
    elif architecture == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
    else:
        raise ValueError("Arquitectura no soportada")

    for param in model.parameters():
        param.requires_grad = False

    if fine_tuning == "fc":
        if architecture.startswith("resnet"):
            for param in model.fc.parameters():
                param.requires_grad = True
        else:
            for param in model.classifier.parameters():
                param.requires_grad = True
    elif fine_tuning == "layer4":
        if not architecture.startswith("resnet"):
            raise ValueError("layer4 aplica a ResNet.")
        for param in model.layer4.parameters():
            param.requires_grad = True
        for param in model.fc.parameters():
            param.requires_grad = True
    elif fine_tuning == "lastfeature":
        if architecture != "efficientnet_b0":
            raise ValueError("lastfeature aplica a EfficientNet en este notebook.")
        for param in model.features[-1].parameters():
            param.requires_grad = True
        for param in model.classifier.parameters():
            param.requires_grad = True
    else:
        raise ValueError("Estrategia de fine-tuning no soportada")

    return model.to(device)

print("Cantidad de clases:", num_classes)


## 13. Pérdida ponderada por desbalance de clases

Como algunos tipos tienen más imágenes que otros, se usa `CrossEntropyLoss` con pesos por clase inversamente proporcionales a la frecuencia.


In [ ]:
counts = train_df["type1"].value_counts()
class_weights = []
for cls in classes:
    class_count = counts[cls]
    weight = len(train_df) / (len(classes) * class_count)
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

for idx, weight in enumerate(class_weights):
    print(f"{idx_to_class[idx]}: {weight.item():.4f}")


## 14. Funciones de entrenamiento y evaluación

Se registra loss y accuracy en train/validación. El mejor modelo de cada experimento se guarda según la mayor accuracy de validación.


In [ ]:
def build_optimizer(model, optimizer_name="adamw", learning_rate=1e-4):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if optimizer_name == "sgd":
        return torch.optim.SGD(trainable_params, lr=learning_rate, momentum=0.9, weight_decay=1e-3)
    if optimizer_name == "adam":
        return torch.optim.Adam(trainable_params, lr=learning_rate, weight_decay=1e-3)
    if optimizer_name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=1e-2)
    raise ValueError("Optimizador no reconocido")

def train_one_experiment(model, experiment_name, learning_rate=1e-4, epochs=5, optimizer_name="adamw", scheduler_name="plateau"):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = build_optimizer(model, optimizer_name=optimizer_name, learning_rate=learning_rate)

    if scheduler_name == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
    else:
        scheduler = None

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_path = DEV_DIR / f"{experiment_name}.pth"

    for epoch in range(epochs):
        model.train()
        train_loss_sum, train_correct, train_total = 0.0, 0, 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * y.size(0)
            train_correct += (logits.argmax(dim=1) == y).sum().item()
            train_total += y.size(0)

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                logits = model(X)
                loss = criterion(logits, y)

                val_loss_sum += loss.item() * y.size(0)
                val_correct += (logits.argmax(dim=1) == y).sum().item()
                val_total += y.size(0)

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        if scheduler is not None:
            scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"{experiment_name} | epoch {epoch+1}/{epochs} | "
            f"lr={current_lr:.2e} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    return history, best_val_acc, best_path

def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            logits = model(X)
            preds = logits.argmax(dim=1).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(y.numpy().tolist())
    return np.array(y_true), np.array(y_pred)


## 15. Experimentación

El enunciado pide comparar al menos 3 configuraciones. Para una prueba rápida se puede usar pocas épocas; para la entrega final conviene aumentar `EPOCHS_PER_EXPERIMENT` según el tiempo disponible en GPU.


In [ ]:
EPOCHS_PER_EXPERIMENT = 5  # Para entrega final pueden aumentar a 20, 30 o 50 según GPU disponible.

experiments = [
    {
        "name": "resnet18_fc_adamw_lr1e3",
        "architecture": "resnet18",
        "fine_tuning": "fc",
        "optimizer": "adamw",
        "lr": 1e-3,
        "scheduler": "plateau",
    },
    {
        "name": "resnet18_layer4_adamw_lr1e4",
        "architecture": "resnet18",
        "fine_tuning": "layer4",
        "optimizer": "adamw",
        "lr": 1e-4,
        "scheduler": "plateau",
    },
    {
        "name": "resnet34_layer4_adamw_lr5e5",
        "architecture": "resnet34",
        "fine_tuning": "layer4",
        "optimizer": "adamw",
        "lr": 5e-5,
        "scheduler": "plateau",
    },
]

results = []
histories = {}

for exp in experiments:
    print("\n" + "=" * 80)
    print("Entrenando:", exp["name"])
    model = create_model(exp["architecture"], exp["fine_tuning"])
    history, best_val_acc, best_path = train_one_experiment(
        model=model,
        experiment_name=exp["name"],
        learning_rate=exp["lr"],
        epochs=EPOCHS_PER_EXPERIMENT,
        optimizer_name=exp["optimizer"],
        scheduler_name=exp["scheduler"],
    )

    histories[exp["name"]] = history
    results.append({
        "experimento": exp["name"],
        "modelo": exp["architecture"],
        "fine_tuning": exp["fine_tuning"],
        "optimizador": exp["optimizer"],
        "learning_rate": exp["lr"],
        "scheduler": exp["scheduler"],
        "epochs": EPOCHS_PER_EXPERIMENT,
        "mejor_val_acc": best_val_acc,
        "mejor_val_loss": min(history["val_loss"]),
        "pesos": best_path.as_posix(),
    })

results_df = pd.DataFrame(results).sort_values("mejor_val_acc", ascending=False)
results_df


## 16. Selección del mejor modelo

Se elige el experimento con mayor accuracy de validación. Sus pesos se copian como `dev/modelo.pth`, que es el nombre pedido para el modelo final.


In [ ]:
best_row = results_df.iloc[0]
best_experiment = best_row["experimento"]
best_architecture = best_row["modelo"]
best_fine_tuning = best_row["fine_tuning"]
best_weights_path = Path(best_row["pesos"])

final_model_path = DEV_DIR / "modelo.pth"
shutil.copy2(best_weights_path, final_model_path)

print("Mejor experimento:", best_experiment)
print("Modelo:", best_architecture)
print("Fine-tuning:", best_fine_tuning)
print("Modelo final guardado en:", final_model_path)


## 17. Curvas de entrenamiento del modelo elegido

Se grafican loss y accuracy de train/validación para analizar overfitting o underfitting.


In [ ]:
best_history = histories[best_experiment]

plt.figure(figsize=(8, 5))
plt.plot(best_history["train_loss"], label="Train loss")
plt.plot(best_history["val_loss"], label="Validation loss")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title(f"Curva de pérdida - {best_experiment}")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(best_history["train_acc"], label="Train accuracy")
plt.plot(best_history["val_acc"], label="Validation accuracy")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title(f"Curva de accuracy - {best_experiment}")
plt.legend()
plt.tight_layout()
plt.show()


## 18. Evaluación final sobre test

El conjunto de test se usa solo al final para estimar el rendimiento real del modelo elegido. Se reporta accuracy global y precision, recall y F1 por clase.


In [ ]:
best_model = create_model(best_architecture, best_fine_tuning)
best_model.load_state_dict(torch.load(final_model_path, map_location=device))
best_model.to(device)

y_true, y_pred = evaluate_model(best_model, test_loader)

test_accuracy = accuracy_score(y_true, y_pred)
print("Accuracy en test:", test_accuracy)

report = classification_report(
    y_true,
    y_pred,
    target_names=classes,
    output_dict=True,
    zero_division=0,
)

metrics_df = pd.DataFrame(report).transpose()
metrics_df


## 19. Matriz de confusión

La matriz de confusión permite ver entre qué tipos se equivoca más el modelo.


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, xticks_rotation=45, values_format="d")
plt.title("Matriz de confusión - test")
plt.tight_layout()
plt.show()


## 20. Análisis de errores

Se muestran ejemplos mal clasificados para comentar posibles causas: similitud visual, imágenes con ruido, poses raras, fondos complejos o clases con pocos ejemplos.


In [ ]:
def collect_misclassified_examples(model, dataset, max_examples=12):
    model.eval()
    errors = []
    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    with torch.no_grad():
        for idx, (X, y) in enumerate(loader):
            X = X.to(device)
            logits = model(X)
            pred = int(logits.argmax(dim=1).cpu().item())
            true = int(y.item())
            if pred != true:
                row = dataset.df.iloc[idx]
                errors.append({
                    "image_path": row["image_path"],
                    "pokemon": row["pokemon"],
                    "real": idx_to_class[true],
                    "predicho": idx_to_class[pred],
                })
            if len(errors) >= max_examples:
                break
    return pd.DataFrame(errors)

errors_df = collect_misclassified_examples(best_model, test_dataset, max_examples=12)
errors_df


In [ ]:
if len(errors_df) > 0:
    cols = 4
    rows = int(np.ceil(len(errors_df) / cols))
    plt.figure(figsize=(14, 3.5 * rows))
    for i, row in errors_df.iterrows():
        image = Image.open(ROOT / row["image_path"]).convert("RGB")
        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(f'{row["pokemon"]}
Real: {row["real"]}
Pred: {row["predicho"]}')
        plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron errores en los primeros ejemplos evaluados del test.")


## 21. Guardado de metadatos para producción

Además de `modelo.pth`, se guarda `class_to_idx.json` para que la app pueda reconstruir el orden de clases al hacer inferencia.


In [ ]:
metadata = {
    "classes": classes,
    "class_to_idx": class_to_idx,
    "idx_to_class": idx_to_class,
    "model_architecture": best_architecture,
    "fine_tuning": best_fine_tuning,
    "image_size": [224, 224],
    "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "test_accuracy": float(test_accuracy),
    "seed": SEED,
}

with open(DEV_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Archivos generados:")
print("-", final_model_path)
print("-", DEV_DIR / "model_metadata.json")
print("-", DATA_DIR / "train.csv")
print("-", DATA_DIR / "val.csv")
print("-", DATA_DIR / "test.csv")


## 22. Conclusión para explicar oralmente

Puntos clave para defender el trabajo:

- Se eligió un problema de clasificación multiclase de imágenes: predecir el tipo principal de un Pokémon.
- Se usó fine-tuning de modelos preentrenados, no uso directo sin modificación.
- La capa final fue reemplazada para la cantidad de clases del dataset.
- El particionado fue estratificado y reproducible con seed fija.
- Las augmentations se aplicaron solo al conjunto de entrenamiento.
- La evaluación final se realizó sobre test, reportando accuracy, precision, recall, F1 y matriz de confusión.
- El modelo final quedó guardado como `dev/modelo.pth` para integrarlo luego con la app.
